In [ ]:
# fetch spectroscopically classified galazies from repository https://github.com/cosmosastro/speczcompilation

from astropy.table import Table

specz = Table.read("../../data/specz_compilation_COSMOS_DR1.1_unique.fits")

In [ ]:
example_galaxy = specz[0]
example_galaxy

In [ ]:
# Use DESI sparcz to fetch the spectrum of the example galaxy
#https://github.com/desihub/tutorials/tree/main

from sparcl.client import SparclClient                                                                                                                           
from astropy.coordinates import SkyCoord                  
import astropy.units as u                                                                                                                                        
import numpy as np  
import matplotlib.pyplot as plt     
from matplotlib import rcParams
rcParams["font.family"] = "Liberation Serif"
rcParams["text.usetex"] = False                                                                                                                                      
                                                                                                                                                                
client = SparclClient(announcement=False)                                                                                                                      
                                                                                                                                                                
def query_desi_spectra(ra=None, dec=None, radius_arcsec=None, z_min=None, z_max=None, limit=5):                                                                               
    """
    Query DESI spectra from SPARCL by cone search + optional redshift range.                                                                                     
                                                                                                                                                                
    Parameters
    ----------                                                                                                                                                   
    ra, dec : float                                                                                                                                            
        Center of search in degrees.
    radius_arcsec : float
        Search radius in arcseconds.                                                                                                                             
    z_min, z_max : float, optional
        Redshift range to filter on.                                                                                                                             
    limit : int                                                                                                                                                
        Max records to return from find().

    Returns
    -------
    list of records, each with .flux, .wavelength, .ra, .dec, .redshift
    """                                                                                                                                                          
    #valid_releases = [r for r in client.get_all_releases() if "DESI" in r]                                                                                           
    constraints = {                                                                                                                                                  
        "data_release": ["DESI-EDR", "DESI-DR1"],
        "spectype": ["QSO"],                                                                                                                                      
    }                                                                                                                                                           
                                                                                                                                                                
    if ra is not None and dec is not None and radius_arcsec is not None:                                                                                       
        r_deg = radius_arcsec / 3600.0
        constraints["ra"]  = [ra - r_deg, ra + r_deg]                                                                                                            
        constraints["dec"] = [dec - r_deg, dec + r_deg]
                                                                                                                                                                
    if z_min is not None and z_max is not None:                                                                                                                  
        constraints["redshift"] = [z_min, z_max]
                                                                                                                                                                
    found = client.find(                                                                                                                                       
        outfields=["sparcl_id", "ra", "dec", "redshift", "spectype", "data_release"],
        constraints=constraints,                                                                                                                                 
        limit=limit,
    )                                                                                                                                                            
                                                                                                                                                                
    if len(found.ids) == 0:                                                                                                                                      
        print("No spectra found.")
        return []                                                                                                                                                
                                                                                                                                                                
    keep_ids = found.ids

    # only do cone refinement if spatial args were given                                                                                                         
    if ra is not None and dec is not None and radius_arcsec is not None:
        center = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)                                                                                                        
        keep_ids = [                                                                                                                                           
            r["sparcl_id"] for r in found.records                                                                                                                
            if center.separation(SkyCoord(ra=r["ra"] * u.deg, dec=r["dec"] * u.deg)).arcsec <= radius_arcsec
        ]                                                                                                                                                        
        if not keep_ids:                                                                                                                                       
            print("No spectra within cone after refinement.")                                                                                                    
            return []                                                                                                                                          

    print(f"Found {len(keep_ids)} spectra, retrieving...")                                                                                                       

    retrieved = client.retrieve(                                                                                                                                 
        uuid_list=keep_ids,                                                                                                                                    
        include=["sparcl_id", "ra", "dec", "redshift", "flux", "wavelength", "ivar", "mask"],
    )                                                                                                                                                            
    return retrieved.records

In [ ]:
# --- example usage ---
records = query_desi_spectra(                                                                                                                                           
    z_min=3.5,                                                                                                                                                
    z_max=3.6,
)                                                                                                                                                                

# access flux/wavelength                                                                                                                                         
for rec in records:                                                                                                                                            
    wave = np.array(rec.wavelength)   # Angstroms
    flux = np.array(rec.flux)         # 1e-17 erg/s/cm2/Angstrom                                                                                                 
    ivar = np.array(rec.ivar)         # inverse variance
    print(f"  z={rec.redshift:.4f}  ra={rec.ra:.5f}  dec={rec.dec:.5f}  npix={len(wave)}") 

In [ ]:
LYMAN_ALPHA_REST = 1216.0  # Angstroms

RUBIN_BANDS = {
    "u": (3200, 4000,  "violet"),
    "g": (4000, 5520,  "royalblue"),
    "r": (5520, 6910,  "green"),
    "i": (6750, 8330,  "goldenrod"),
    "z": (8030, 9280,  "orange"),
    "y": (9280, 10800, "tomato"),
}

def plot_spectrum(rec, show_rubin_bands=True):  
    wave = np.array(rec.wavelength)
    flux = np.array(rec.flux)
    ivar = np.array(rec.ivar)
    z = rec.redshift

    err = np.where(ivar > 0, 1.0 / np.sqrt(ivar), np.nan)

    fig, ax = plt.subplots(figsize=(12, 3))

    if show_rubin_bands:
        for band, (lam_lo, lam_hi, color) in RUBIN_BANDS.items():
            ax.axvspan(lam_lo, lam_hi, alpha=0.08, color=color)

    ax.plot(wave, flux, lw=0.6, color="steelblue")
    ax.fill_between(wave, flux - err, flux + err, alpha=0.3, color="steelblue")
    ax.axhline(0, color="k", lw=0.5, ls="--")

    ax.axvline(LYMAN_ALPHA_REST, color="gray", lw=1.0, ls="--", label=f"Lyα rest ({LYMAN_ALPHA_REST:.0f} Å)")
    ax.axvline(LYMAN_ALPHA_REST * (1 + z), color="tomato", lw=1.0, ls="--", label=f"Lyα redshifted ({LYMAN_ALPHA_REST * (1 + z):.0f} Å)")

    ax.set_xlabel("Wavelength (Å)")
    ax.set_ylabel(r"Flux ($10^{-17}$ erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)")
    ax.set_title(f"z={z:.4f}  ra={rec.ra:.5f}  dec={rec.dec:.5f}")
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_spectrum(records[4])

In [ ]:
client.all_datasets

In [ ]:
client.get_all_databases()

In [ ]:
# grab uv spectra from HST

from astroquery.mast import Observations
                                                                                                                                                                
obs = Observations.query_region(                                                                                                                               
    "150.1 2.2",
    radius="30s",                                                                                                                                                
)
                                                                                                                                                                
# filter to UV spectroscopy                                                                                                                                    
uv_spec = obs[
    (obs["obs_collection"] == "HST") &                                                                                                                           
    (obs["dataproduct_type"] == "spectrum") &
    (obs["instrument_name"].astype(str).str.contains("COS|STIS"))                                                                                                
]                                                                                                                                                                
uv_spec["obs_id", "instrument_name", "filters", "t_min", "t_max"]
                                                                                                                                                                
products = Observations.get_product_list(uv_spec)                                                                                                                
Observations.download_products(products, productType="SCIENCE") 

In [ ]:
# see where different spectra fall on color-color plot

In [ ]:
# fetch rubin images from this position

In [ ]:
# psf fit to get rubin colors for this 